# RAG系统的评估体系（RAG Triad/Ragas）与高级Query增强策略（HyDE & Multi-Query）。
在企业级落地中，我们会面临两个终极问题：

1. “怎么证明 RAG 变好了还是变差了？”（如何对 RAG 系统进行量化评估与诊断？）
2. “当用户输入表达不清、简短或语义跨度很大时，检索怎么破局？”（高级 RAG 策略：Query 变换与生成增强）

## 量化诊断 RAG 质量，Query 增强与高级 RAG 策略
1. RAG 评估三元组（RAG Triad）：掌握 Context Relevance（上下文相关性）、Faithfulness（忠实度/幻觉指标） 和 Answer Relevance（回答相关性） 的量化诊断公式与 LLM-as-a-Judge 评价机制。
2. Query 变形与假想文档生成（HyDE & Multi-Query）：理解为什么“用 Query 找 Doc”不如“用 Doc 找 Doc”，掌握 HyDE（Hypothetical Document Embeddings）和多查询扩展（Multi-Query Expansion）的实现。
3. GraphRAG 与高级架构全景：理解传统 Vector RAG 在处理“全局总结性问题”时的局限，以及知识图谱 GraphRAG 的适用场景。

## 评估闭环与 Query 增强
1. RAG 评估三元组 (RAG Triad)
    评估传统的机器学习通常依赖 Ground Truth（标准答案），但 RAG 的生成内容极具灵活性。业界（如 Ragas, TruLens 框架）提出了 RAG Triad（评估三元组），将问题精准拆解为 3 个独立维度：
   |评估维度|考察对象|核心问题|错误诊断与优化方向|
   |:----|:------|:-----|:------|
   |1. Context Relevance(上下文相关性)|Query $\leftrightarrow$ Context|检索出来的文档块，是否真的包含了回答问题所需的信息？|检索不准/噪声过多。优化方案：调整 Chunk 大小、增强 Hybrid Search/Reranker。|
   |2. Faithfulness(忠实度 / 幻觉度)|Context $\leftrightarrow$ Response|LLM 生成的回答，是否完全基于给定的 Context，有没有张冠李戴或编造？|模型幻觉/未遵循上下文。优化方案：调整 Prompt 约束、降低 Temperature、更换生成模型。|
   |3. Answer Relevance(回答相关性)|Query $\leftrightarrow$ Response|LLM 的回答是否真正解答了用户的原始 Query？有没有答非所问？|偏题/漏答/逻辑混乱。优化方案：优化 System Prompt 结构、使用 Chain-of-Thought (CoT)。|
   > LLM-as-a-Judge（以模型评估模型）：在生产实践中，我们通常使用能力更强的模型（如 GPT-4 / Claude-3.5）作为裁判，利用结构化 Prompt 对上述三指标进行自动打分（0-1分），实现自动化回归测试。

2. Query 增强策略：HyDE 与 Multi-Query

    用户的 Query 往往极其简短或口语化（如“ERR_9021 怎么搞”），而知识库文档通常是长篇大论规范书。Short Query 与 Long Document 在 Embedding 空间中天然存在语义不对称。
   为了弥补这一差距，高级 RAG 引入了检索前（Pre-Retrieval）增强：
   1. HyDE (Hypothetical Document Embeddings，假想文档嵌入)
      * 核心思路：先让 LLM 根据用户的 Query 凭空“瞎编/假想”一份解答文档（Hypothetical Document），然后再拿这份假想文档的向量去数据库里检索真实文档！
      * 为什么有效：假想文档虽然可能包含事实错误，但它的文本结构、专业术语和上下文分布与真实文档极度相似，实现了从“文档到文档”的高维向量对齐

    2. Multi-Query / Sub-Query (多查询扩展与子查询拆解)
       * 多查询扩展（Multi-Query）：让 LLM 从不同视角将 Prompt 改写成 3~5 个同义 Query，分别进行检索后再融合结果。
       * 复杂问题拆解（Sub-Query）：对于“对比 Q1 和 Q2 的产品架构”，LLM 拆解为“Q1 的产品架构”与“Q2 的产品架构”两条子链，分别检索后拼接上下文。

#### HyDE 与 LLM 评估诊断流
下面的 Python 代码完整展示了 HyDE 假设文档生成 以及 LLM-as-a-Judge 评估打分逻辑：

In [ ]:
import numpy as np
from typing import List, Dict

# --- 模拟知识库环境 ---
knowledge_base = {
    "doc_1": "错误码 ERR_9021 表示 Redis 连接池爆满。处理方案：检查 max_connections 参数，增加连接池上限或关闭未释放的客户端连接。",
    "doc_2": " Agent 规划模块通常采用 ReAct 框架，将思考 (Thought) 与行动 (Action) 结合形成循环过程。",
    "doc_3": "混合检索 (Hybrid Search) 通过组合 Dense Vector 与 Sparse BM25，显著提升精确专有名词的召回率。"
}

# --- 1. HyDE (Hypothetical Document Embeddings) 实战 ---
def generate_hypothetical_document(query: str) -> str:
    """
    第一步：让 LLM 假想生成一份规范文档（模拟 LLM 输出）
    """
    print(f"🤖 [HyDE 触发] 正在针对 Query: '{query}' 生成假想解答文档...")
    # 模拟 LLM 根据 Query 生成的假想文本
    if "ERR_9021" in query:
        hypo_doc = "针对系统报错 ERR_9021，通常与数据库或 Redis 连接池配置相关。处理步骤包括排查连接池泄露、调整连接数配置并重启服务。"
    else:
        hypo_doc = f"关于{query}的详细处理流程与系统架构说明。"

    return hypo_doc

def mock_vector_search_with_text(text_to_embed: str) -> str:
    """
    第二步：使用假想文档去匹配知识库（比用原始 Query 语义更契合）
    """
    print(f"🔎 [向量检索] 使用输入语义提取关键特征并在向量库中寻优...")
    if "连接池" in text_to_embed or "ERR_9021" in text_to_embed:
        return knowledge_base["doc_1"]
    return knowledge_base["doc_2"]

# --- 2. RAG Triad 自动化评估模块 (LLM-as-a-Judge) ---
def evaluate_rag_triad(query: str, retrieved_context: str, generated_answer: str) -> Dict[str, float]:
    """
    基于 RAG 三元组指标进行结构化评估计算 (0.0 - 1.0)
    """
    print("\n📊 --- 开始执行 RAG Triad 评估诊断 ---")

    # 1. Context Relevance (Query vs. Context)
    context_rel = 0.0
    if "ERR_9021" in query and "ERR_9021" in retrieved_context:
        context_rel = 0.95
    elif "ERR_9021" in query and "连接池" in retrieved_context:
        context_rel = 0.70

    # 2. Faithfulness (Context vs. Answer)
    # 检查回答内容是否超越了 Context（模拟幻觉检测）
    faithfulness = 1.0
    if "重启服务器" in generated_answer and "重启服务器" not in retrieved_context:
        faithfulness -= 0.3  # 惩罚幻觉内容

    # 3. Answer Relevance (Query vs. Answer)
    answer_rel = 0.0
    if "连接池" in generated_answer or "ERR_9021" in generated_answer:
        answer_rel = 0.90

    return {
        "Context Relevance (上下文相关性)": context_rel,
        "Faithfulness (忠实度/无幻觉)": faithfulness,
        "Answer Relevance (回答相关性)": answer_rel
    }

# --- 3. 端到端完整流程运行 ---
if __name__ == "__main__":
    raw_user_query = "ERR_9021 报错"
    print(f"📥 原始用户 Query: '{raw_user_query}'\n")

    # 步骤 A: 运行 HyDE 生成假想文档
    hypo_doc = generate_hypothetical_document(raw_user_query)
    print(f"📄 [生成假想文档 HyDE]: {hypo_doc}\n")

    # 步骤 B: 拿着假想文档去检索真实 Context
    retrieved_context = mock_vector_search_with_text(hypo_doc)
    print(f"📚 [检索获得的 Context]: {retrieved_context}\n")

    # 步骤 C: 大模型基于 Context 生成最终回答（模拟带有一点超纲生成的回答）
    generated_answer = "遇到 ERR_9021 报错时，请检查 Redis connection pool 的 max_connections 配置并清理未释放连接。如果不行，请重启服务器。"
    print(f"💡 [LLM 最终回答]: {generated_answer}")

    # 步骤 D: 评估诊断
    scores = evaluate_rag_triad(raw_user_query, retrieved_context, generated_answer)
    for metric, score in scores.items():
        print(f"  • {metric}: {score:.2f}")

1. HyDE 的适用边界与性能代价：
    * HyDE 需要先调用一次大模型生成假想文档，再进行向量检索。这增加了一次完整的 LLM API 延迟（首包时间增加 500ms~2s）。
    * 工程思考：在实时性要求极高（如客服实时弹窗助手）的场景中，你会采取哪些策略来降低 HyDE 的延迟？如果 LLM 产生的“假想文档”产生了严重的幻觉偏见，会导致检索彻底跑偏吗？如何防护？
2. GraphRAG 的架构取舍：
    * 传统的 Vector RAG 在面对“这本 50 页报告里提到的主要风险点有哪些？”（全局总结性问题）时表现极差，因为向量只能搜索特定小 Block。
    * 思考与拓展：结合微软开源的 GraphRAG 思想，为什么“抽取实体-关系建立知识图谱”能够弥补向量数据库在全局总结、跨文档联动上的缺陷？